In [1]:
# Téléchargement et extraction de GloVe en ligne de commande (Colab)
!wget http://nlp.stanford.edu/data/glove.6B.zip
!unzip -q glove.6B.zip

--2026-05-18 09:29:15--  http://nlp.stanford.edu/data/glove.6B.zip
Resolving nlp.stanford.edu (nlp.stanford.edu)... 171.64.67.140
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:80... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://nlp.stanford.edu/data/glove.6B.zip [following]
--2026-05-18 09:29:15--  https://nlp.stanford.edu/data/glove.6B.zip
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip [following]
--2026-05-18 09:29:15--  https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip
Resolving downloads.cs.stanford.edu (downloads.cs.stanford.edu)... 171.64.64.22
Connecting to downloads.cs.stanford.edu (downloads.cs.stanford.edu)|171.64.64.22|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 862182613 (822M) [application/zip]
Saving to: ‘glove.6B.zip’

glov

In [4]:
# 1. Suppression des éventuels résidus de fichiers corrompus
!rm -f Sarcasm_Headlines_Dataset.json.zip

# 2. Téléchargement depuis un miroir alternatif très stable (dépôt brut GitHub)
print("📥 Téléchargement du dataset de sarcasme...")
!wget -q https://raw.githubusercontent.com/L some-repo-mirror/Sarcasm-Detection/master/Sarcasm_Headlines_Dataset.json -O Sarcasm_Headlines_Dataset.json

# Si le premier lien échoue, voici le plan B automatique (format compressé officiel)
import os
if os.path.getsize('Sarcasm_Headlines_Dataset.json') < 1000: # Si le fichier est vide ou erreur 404
    print("🔄 Plan B : Tentative via le second miroir...")
    !wget -q https://storage.googleapis.com/download.tensorflow.org/data/sarcasm.json -O Sarcasm_Headlines_Dataset.json

print("✅ Vérification du fichier :")
!ls -lh Sarcasm_Headlines_Dataset.json

📥 Téléchargement du dataset de sarcasme...
🔄 Plan B : Tentative via le second miroir...
✅ Vérification du fichier :
-rw-r--r-- 1 root root 5.4M Feb 20  2020 Sarcasm_Headlines_Dataset.json


In [7]:
import json
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# ==========================================
# 1. CHARGEMENT ROBUSTE DES DONNÉES
# ==========================================
sentences = []
labels = []

try:
    # Option A : Format standard de TensorFlow (le miroir Google)
    with open('Sarcasm_Headlines_Dataset.json', 'r', encoding='utf-8') as f:
        data = json.load(f)
        for item in data:
            sentences.append(item['headline'])
            labels.append(item['is_sarcastic'])
except json.JSONDecodeError:
    # Option B : Format JSON Lines d'origine (Kaggle)
    with open('Sarcasm_Headlines_Dataset.json', 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                item = json.loads(line)
                sentences.append(item['headline'])
                labels.append(item['is_sarcastic'])

print(f"📊 Dataset chargé avec succès ! Nombre d'exemples : {len(sentences)}")

# Séparation Train / Test (80% / 20%)
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    sentences, np.array(labels), test_size=0.2, random_state=42
)


# ==========================================
# 2. TOKENIZATION ET PADDING
# ==========================================
VOCAB_SIZE = 20000
MAX_LEN = 40  # Les titres de presse sont courts, 40 mots suffisent
EMBED_DIM = 100

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train_raw)

X_train = pad_sequences(tokenizer.texts_to_sequences(X_train_raw), maxlen=MAX_LEN, padding='post')
X_test = pad_sequences(tokenizer.texts_to_sequences(X_test_raw), maxlen=MAX_LEN, padding='post')


import os

# ==========================================
# 3. EXTRACTION UNIQUEMENT DE GLOVE 100D (CORRIGÉ)
# ==========================================
if not os.path.exists('glove.6B.100d.txt'):
    print("📂 Extraction du fichier GloVe 100d depuis l'archive zip...")
    !unzip -q glove.6B.zip glove.6B.100d.txt

print("⚙️ Alignement de la matrice GloVe avec notre vocabulaire...")
embeddings_index = {}
with open('glove.6B.100d.txt', 'r', encoding='utf-8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        coefs = np.asarray(values[1:], dtype='float32')
        embeddings_index[word] = coefs

# Initialisation de notre matrice finale avec des zéros
embedding_matrix = np.zeros((VOCAB_SIZE, EMBED_DIM))
for word, i in tokenizer.word_index.items():
    if i < VOCAB_SIZE:
        embedding_vector = embeddings_index.get(word)
        if embedding_vector is not None:
            embedding_matrix[i] = embedding_vector


# ==========================================
# 4. ARCHITECTURE DU MODÈLE BI-LSTM
# ==========================================
model = keras.Sequential([
    layers.Input(shape=(MAX_LEN,)),
    layers.Embedding(
        VOCAB_SIZE,
        EMBED_DIM,
        embeddings_initializer=keras.initializers.Constant(embedding_matrix),
        trainable=False
    ),
    layers.Bidirectional(layers.LSTM(64, return_sequences=True)),
    layers.Bidirectional(layers.LSTM(32)),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()


# ==========================================
# 5. ENTRAÎNEMENT
# ==========================================
print("🚀 Début de l'entraînement...")
history = model.fit(
    X_train, y_train,
    epochs=6,
    batch_size=64,
    validation_split=0.1
)


# ==========================================
# 6. ÉVALUATION ET RAPPORT DE PERFORMANCES
# ==========================================
y_pred_probs = model.predict(X_test, verbose=0)
y_pred = (y_pred_probs > 0.5).astype("int32")

print("\n🧾 Rapport de classification final :")
print(classification_report(y_test, y_pred, target_names=['Real News', 'Sarcastic']))

model.save('sarcasm_detector.keras')
print("💾 Modèle sauvegardé avec succès sous 'sarcasm_detector.keras'")


# ==========================================
# 7. ZONE DE PRÉDICTION INTERACTIVE
# ==========================================
def predict_sarcasm(headline):
    seq = tokenizer.texts_to_sequences([headline])
    padded = pad_sequences(seq, maxlen=MAX_LEN, padding='post')
    proba = model.predict(padded, verbose=0)[0][0]

    if proba > 0.5:
        print(f"🔹 Titre : '{headline}'\n🔸 Verdict : 🤔 SARCASTIC ({proba*100:.1f}% de confiance)\n")
    else:
        print(f"🔹 Titre : '{headline}'\n🔸 Verdict : 📰 REAL NEWS ({(1-proba)*100:.1f}% de confiance)\n")

print("\n🔮 Test des prédictions à la volée :\n")
predict_sarcasm("Local man specialized in doing absolutely nothing receives national award")
predict_sarcasm("US Government announces new economic plan to tackle inflation")

📊 Dataset chargé avec succès ! Nombre d'exemples : 26709
⚙️ Alignement de la matrice GloVe avec notre vocabulaire...


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 40, 100)        │     2,000,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 40, 128)        │        84,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 64)             │        41,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,127,809 (8.12 MB)

 Trainable params: 127,809 (499.25 KB)

 Non-trainable params: 2,000,000 (7.63 MB)

🚀 Début de l'entraînement...
Epoch 1/6
301/301 ━━━━━━━━━━━━━━━━━━━━ 12s 17ms/step - accuracy: 0.7487 - loss: 0.5069 - val_accuracy: 0.8203 - val_loss: 0.4050
Epoch 2/6
301/301 ━━━━━━━━━━━━━━━━━━━━ 4s 14ms/step - accuracy: 0.8304 - loss: 0.3900 - val_accuracy: 0.8456 - val_loss: 0.3500
Epoch 3/6
301/301 ━━━━━━━━━━━━━━━━━━━━ 6s 19ms/step - accuracy: 0.8590 - loss: 0.3287 - val_accuracy: 0.8465 - val_loss: 0.3341
Epoch 4/6
301/301 ━━━━━━━━━━━━━━━━━━━━ 9s 14ms/step - accuracy: 0.8754 - loss: 0.2960 - val_accuracy: 0.8577 - val_loss: 0.3345
Epoch 5/6
301/301 ━━━━━━━━━━━━━━━━━━━━ 6s 19ms/step - accuracy: 0.8943 - loss: 0.2592 - val_accuracy: 0.8704 - val_loss: 0.3097
Epoch 6/6
301/301 ━━━━━━━━━━━━━━━━━━━━ 4s 14ms/step - accuracy: 0.9115 - loss: 0.2257 - val_accuracy: 0.8591 - val_loss: 0.3417

🧾 Rapport de classification final :
              precision    recall  f1-score   support

   Real News       0.82      0.94      0.87      2996
   Sarcastic       0.90      0.73      0.81      2346

 